# March Machine Learning Mania 2026

|No| Name  | Notebook |
| --- | ------ | ------- |
| 00 | Dataset |  https://www.kaggle.com/competitions/march-machine-learning-mania-2026/data |
| 01 | Understand Data Structure  | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p1-dataset-overview-structure-understanding/ |
| 02 | Create a baseline model only on Male rqd. 4 datasets | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p2-basline-male-4-datasets/ |
| 03 | Features Eng | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p3-features-eng/ |
| 04 | Simple ELO | https://www.kaggle.com/code/rudraprasadbhuyan/ncaa-p4-simple-elo/ |
| | |  |

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

**Story**

Features

- Part 2 V1 - Baseline model (`win diff`, `seed diff`)
- Part 3 
  - V1 - `margin diff`
  - V2 - `recent win pct diff` Hypothesis: Teams playing well recently perform better than tournament
- Part 4
  - V1: Simple ELO + All Features
  - V2: Remove these 'Margin_Diff','RecentWinPct_Diff'still good result.

# Basic 

> - Again, without understanding the basic we can't code.
> - First, I am new to sports analysis.
> - So we need info and data to understand the game rules.
> - So if you new let's take the help of an infinitely intelligent


## What is ELO?

ELO is a dynamic rating system.

* Every team starts with same rating (ex: 1500)
* If strong team beats weak team → small rating change
* If weak team upsets strong team → big rating change

It automatically captures:

* strength
* upset power
* consistency
* schedule difficulty



## ELO Formula
- wiki: https://en.wikipedia.org/wiki/Elo_rating_system

For a match:

**Expected score:**

$$E_A = 1 / (1 + 10^{(R_B - R_A)/400}) $$

**Update:**

$$R_A = R_A + K * (S_A - E_A)$$
Where:

* `R_A` = rating of Team A
* `R_B` = rating of Team B
* `S_A` = 1 if win, 0 if loss
* `K` = learning rate (usually 20)

---


# Import Data

In [58]:
# Root file
data_file_path = r"/kaggle/input/march-machine-learning-mania-2026"
# Root file
data_file_path = r"C:\Users\Rudra\Desktop\kaggle\NCAA\data"

# Load only required files
regular = pd.read_csv(os.path.join(data_file_path, "MRegularSeasonCompactResults.csv"))
tourney = pd.read_csv(os.path.join(data_file_path,"MNCAATourneyCompactResults.csv"))
seeds = pd.read_csv(os.path.join(data_file_path,"MNCAATourneySeeds.csv"))
submission = pd.read_csv(os.path.join(data_file_path,"SampleSubmissionStage1.csv"))

# Sample View
display(regular.sample(3))
display(tourney.sample(3))
display(seeds.sample(3))
display(submission.sample(3))

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
86748,2005,103,1293,82,1122,73,H,0
183536,2024,45,1229,85,1369,64,H,0
21945,1990,93,1301,84,1438,58,H,0


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
120,1986,145,1261,59,1246,57,N,0
1794,2013,136,1272,54,1388,52,N,0
1460,2008,136,1266,74,1246,66,N,0


,Season,Seed,TeamID
1825,2013,W16a,1241
1202,2003,Y16,1237
2238,2019,X03,1403


,ID,Pred
301786,2024_1263_1469,0.5
92368,2022_3199_3464,0.5
83025,2022_3166_3356,0.5


# clean Datasets

In [59]:
wins = regular.groupby(['Season', 'WTeamID']).size().reset_index(name='wins')
losses = regular.groupby(['Season', 'LTeamID']).size().reset_index(name='losses')

wins = wins.rename(columns={"WTeamID": 'TeamID'})
losses = losses.rename(columns={'LTeamID': 'TeamID'})

team_stats = pd.merge(
    wins, losses,
    how="outer", on=["Season", 'TeamID']
).fillna(0)

team_stats["TotalGames"] = team_stats["wins"] + team_stats["losses"]
team_stats['WinPct'] = team_stats['wins'] / team_stats['TotalGames']

display(team_stats.sample(3))


# Clean Seeds
seeds['SeedNum'] = seeds['Seed'].str[1:3].astype(int)
display(seeds.sample(3))


# Clean Tournament
tourney['Team1'] = tourney[['WTeamID', 'LTeamID']].min(axis=1)
tourney['Team2'] = tourney[['WTeamID', 'LTeamID']].max(axis=1)
tourney['Team1Win'] = (tourney['WTeamID'] == tourney['Team1']).astype(int)

train = tourney[['Season', 'Team1', 'Team2', 'Team1Win']]
display(train.sample(3))
print(train.shape)

,Season,TeamID,wins,losses,TotalGames,WinPct
10700,2018,1271,5.0,25.0,30.0,0.166667
1054,1988,1352,11.0,14.0,25.0,0.440000
4217,1999,1193,12.0,15.0,27.0,0.444444


,Season,Seed,TeamID,SeedNum
482,1992,Y03,1116,3
502,1992,Z07,1385,7
2498,2024,W09,1321,9


,Season,Team1,Team2,Team1Win
1372,2006,1207,1326,1
1734,2012,1285,1458,0
167,1987,1196,1345,1


(2585, 4)


# Add Features

# ELO Calculator

In [60]:
def calculate_elo(df, k=20, initial_rating=1500):
    df = df.sort_values(['Season', 'DayNum'])
    
    final_elo = []  # store final ratings

    for season in df['Season'].unique():
        
        season_games = df[df['Season'] == season]
        teams = pd.unique(season_games[['WTeamID','LTeamID']].values.ravel())
        
        # initialize ratings
        ratings = {team: initial_rating for team in teams}
        
        for _, row in season_games.iterrows():
            teamA = row['WTeamID']
            teamB = row['LTeamID']
            
            RA = ratings[teamA]
            RB = ratings[teamB]
            
            # expected scores
            EA = 1 / (1 + 10 ** ((RB - RA) / 400))
            EB = 1 - EA
            
            # update ratings
            ratings[teamA] = RA + k * (1 - EA)
            ratings[teamB] = RB + k * (0 - EB)
        
        # save final ratings
        for team, rating in ratings.items():
            final_elo.append([season, team, rating])
    
    elo_df = pd.DataFrame(final_elo, columns=['Season','TeamID','ELO'])
    return elo_df

In [61]:
elo_ratings = calculate_elo(regular)

In [62]:
display(elo_ratings.sample(3))

,Season,TeamID,ELO
12799,2024,1299,1367.919206
922,1988,1274,1519.538750
4422,1999,1354,1523.150925


In [63]:
def add_feature_elo(df):
    # team1 elo
    df = df.merge(
        elo_ratings,
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'ELO':'Team1_ELO'}).drop('TeamID', axis=1)

    # team2 elo
    df = df.merge(
        elo_ratings,
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'ELO':'Team2_ELO'}).drop('TeamID', axis=1)
    
    return df

## Add Score Margin

In [64]:
regular['ScoreDiff'] = regular['WScore'] - regular['LScore']

# For Winners
w_margin = regular.groupby(['Season', 'WTeamID'])['ScoreDiff'].mean().reset_index()
w_margin = w_margin.rename(columns={'WTeamID':'TeamID', 'ScoreDiff':'AvgMargin'})

# For Losers
l_margin = regular.groupby(['Season','LTeamID'])['ScoreDiff'].mean().reset_index()
l_margin['ScoreDiff'] = -l_margin['ScoreDiff']
l_margin = l_margin.rename(columns={'LTeamID':'TeamID','ScoreDiff':'AvgMargin'})

# Combine
margin = pd.concat([w_margin, l_margin])
margin = margin.groupby(['Season','TeamID'])['AvgMargin'].mean().reset_index()
display(margin.sample(3))

,Season,TeamID,AvgMargin
4366,1999,1372,-2.598485
6854,2007,1245,2.427273
2590,1993,1429,0.275735


## Add Recent Win Pct

In [65]:
regular_recent =  regular[regular['DayNum'] > 100].copy()

# wins
wins_recent = regular_recent.groupby(['Season', 'WTeamID']).size().reset_index(name='Wins')
wins_recent = wins_recent.rename(columns={'WTeamID': 'TeamID'})

# Losses
losses_recent = regular_recent.groupby(['Season', 'LTeamID']).size().reset_index(name='Losses')
losses_recent = losses_recent.rename(columns={'LTeamID': 'TeamID'})

recent_stats = pd.merge(
    wins_recent, losses_recent,
    on=['Season', 'TeamID'],
    how='outer'
).fillna(0)

recent_stats['Total'] = recent_stats['Wins'] + recent_stats['Losses']
recent_stats['RecentWinPct'] =  recent_stats['Wins'] / recent_stats['Total']

recent_stats.head(3)


,Season,TeamID,Wins,Losses,Total,RecentWinPct
0,1985,1102,3.0,5.0,8.0,0.375000
1,1985,1103,3.0,4.0,7.0,0.428571
2,1985,1104,7.0,2.0,9.0,0.777778


In [66]:
def add_feature_recent_win_pct(df):
    # recent team 1 win pct
    df = df.merge(
        recent_stats[['Season','TeamID','RecentWinPct']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'RecentWinPct':'Team1_RecentWinPct'}).drop('TeamID', axis=1)

    # recent team 2 win pct
    df = df.merge(
        recent_stats[['Season','TeamID','RecentWinPct']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'RecentWinPct':'Team2_RecentWinPct'}).drop('TeamID', axis=1)


    return df

## Add Win Pct Diff 

In [67]:
def add_feature_win_pct(df:pd.date_range) -> pd.DataFrame:
    """Create features for both train and test(submission) data."""

    # Team1 win pct
    df = df.merge(
        team_stats[['Season','TeamID','WinPct']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'WinPct':'Team1_WinPct'}).drop('TeamID', axis=1)

    # Team2 win pct
    df = df.merge(
        team_stats[['Season','TeamID','WinPct']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'WinPct':'Team2_WinPct'}).drop('TeamID', axis=1)
    
    return df

## Add Seed

In [68]:

def add_feature_seed(df):
    # team1 seed
    df = df.merge(
        seeds[['Season','TeamID','SeedNum']],
        left_on=['Season','Team1'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'SeedNum':'Team1_Seed'}).drop('TeamID', axis=1)

    # team 2 seed
    df = df.merge(
        seeds[['Season','TeamID','SeedNum']],
        left_on=['Season','Team2'],
        right_on=['Season','TeamID'],
        how='left'
    ).rename(columns={'SeedNum':'Team2_Seed'}).drop('TeamID', axis=1)
    
    return df

## Add Avg Margin

In [69]:
def add_feature_avg_margin(df):
    # team 1 avg margin
    df = df.merge(
        margin,
        left_on=['Season', 'Team1'],
        right_on=['Season', 'TeamID'],
        how='left',
    ).rename(columns={'AvgMargin': 'Team1_AvgMargin'}).drop('TeamID', axis=1)
    
    # team 2 avg margin
    df = df.merge(
        margin,
        left_on=['Season', 'Team2'],
        right_on=['Season', 'TeamID'],
        how='left',
    ).rename(columns={'AvgMargin': 'Team2_AvgMargin'}).drop('TeamID', axis=1)

    return df

In [70]:
def add_all_features(df):
    """Call al the features function at a time."""
    df = add_feature_win_pct(df)
    df = add_feature_seed(df)
    df = add_feature_avg_margin(df)
    df = add_feature_recent_win_pct(df)
    df = add_feature_elo(df)
    
    return df

In [71]:
train = add_all_features(train)

# Create Model Inputs Features

In [72]:
def create_diff(df:pd.DataFrame) ->pd.DataFrame:
    """ Create Features difference."""
    df['WinPct_Diff'] = df['Team1_WinPct'] - df['Team2_WinPct']

    df['Seed_Diff'] = df['Team2_Seed'] - df['Team1_Seed'] 

    df['Margin_Diff'] = df['Team1_AvgMargin'] - df['Team2_AvgMargin']
    
    df['RecentWinPct_Diff'] = df['Team1_RecentWinPct'] - df['Team2_RecentWinPct']

    df['ELO_Diff'] =  df['Team1_ELO'] - df['Team2_ELO']
    
    return df

In [73]:
train = create_diff(train)

train.sample(3)

,Season,Team1,Team2,Team1Win,Team1_WinPct,Team2_WinPct,Team1_Seed,Team2_Seed,Team1_AvgMargin,Team2_AvgMargin,Team1_RecentWinPct,Team2_RecentWinPct,Team1_ELO,Team2_ELO,WinPct_Diff,Seed_Diff,Margin_Diff,RecentWinPct_Diff,ELO_Diff
1552,2009,1340,1462,0,0.709677,0.781250,13,4,1.606061,3.500000,0.75,0.625000,1576.663352,1618.571368,-0.071573,-9,-1.893939,0.125000,-41.908016
665,1995,1246,1408,1,0.862069,0.709677,1,9,9.670000,0.858586,0.90,0.555556,1672.428702,1592.278030,0.152392,8,8.811414,0.344444,80.150672
1080,2002,1274,1281,0,0.774194,0.656250,5,12,2.669643,3.170996,0.50,0.444444,1619.514692,1568.533706,0.117944,7,-0.501353,0.055556,50.980986


# Model

In [74]:
train.columns

Index(['Season', 'Team1', 'Team2', 'Team1Win', 'Team1_WinPct', 'Team2_WinPct',
       'Team1_Seed', 'Team2_Seed', 'Team1_AvgMargin', 'Team2_AvgMargin',
       'Team1_RecentWinPct', 'Team2_RecentWinPct', 'Team1_ELO', 'Team2_ELO',
       'WinPct_Diff', 'Seed_Diff', 'Margin_Diff', 'RecentWinPct_Diff',
       'ELO_Diff'],
      dtype='object')

In [75]:
model_train_input_cols = ['WinPct_Diff','Seed_Diff','Margin_Diff','RecentWinPct_Diff', 'ELO_Diff']

In [76]:
X = train[model_train_input_cols]
y = train['Team1Win']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

preds = model.predict_proba(X_val)[:, 1]

print(f'LogLoss {log_loss(y_val, preds)}')

LogLoss 0.5231240561866781


# Submission

In [ ]:
def create_id(df):
    df[['Season','Team1','Team2']] = df['ID'].str.split('_', expand=True)

    df['Season'] = df['Season'].astype(int)
    df['Team1'] = df['Team1'].astype(int)
    df['Team2'] = df['Team2'].astype(int)

    return df

submission = create_id(submission)
submission = add_all_features(submission)

# only men for now
submission_men = submission[
    (submission['Team1'] < 2000) &
    (submission['Team2'] < 2000)
].copy()

submission = create_diff(submission)

submission[model_train_input_cols] = submission[model_train_input_cols].fillna(0)
# submission[['WinPct_Diff','Seed_Diff', 'Margin_Diff', 'RecentWinPct_Diff']] = \
# submission[['WinPct_Diff','Seed_Diff', 'Margin_Diff', 'RecentWinPct_Diff']].fillna(0)

# Predict
submission['Pred'] = model.predict_proba(
    submission[model_train_input_cols]
)[:,1]

display(submission[['ID','Pred']])

# Export
submission[['ID','Pred']].to_csv("submission.csv", index=False)
print('submission.csv exported')

,ID,Pred
0,2022_1101_1102,0.676564
1,2022_1101_1103,0.460058
2,2022_1101_1104,0.456780
3,2022_1101_1105,0.749935
4,2022_1101_1106,0.726857
...,...,...
519139,2025_3477_3479,0.496032
519140,2025_3477_3480,0.496032
519141,2025_3478_3479,0.496032
519142,2025_3478_3480,0.496032


submission.csv exported
